# 02 - EDA & Data Preprocessing

Exploratory Data Analysis (EDA) on the crawled dataset.

This notebook analyzes:
- **Distribution**: Images per class (balance check)
- **Samples**: Visual inspection of images
- **Size**: Width/height distribution
- **Quality**: Basic stats (before preprocessing)

In [ ]:
import os
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

# Dataset structure: dataset_v2\\raw\\<ClassName>\\*.jpg
dataset_root = Path('..') / 'dataset_v2' / 'raw'

# 16 classes from project specification
import sys
sys.path.append('..')
from src.config.disease_classes import CLASS_NAMES as classes

print(f'Dataset root: {dataset_root}')
print(f'Classes: {len(classes)}')

## 1. Class Distribution

In [ ]:
# Count images per class
data = []
for cls in classes:
    path = dataset_root / cls
    if path.exists():
        num_images = len([f for f in os.listdir(path) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))])
    else:
        num_images = 0
    data.append({'Class': cls, 'Count': num_images})

df_count = pd.DataFrame(data)
print(f'\nTotal images: {df_count["Count"].sum()} / 10000 (Target)')
print(df_count.to_string())

In [ ]:
# Bar plot - distribution
plt.figure(figsize=(14, 6))
import matplotlib.cm as cm
import numpy as np
colors = cm.rainbow(np.linspace(0, 1, len(df_count)))
bars = plt.bar(range(len(df_count)), df_count['Count'], color=colors)
plt.axhline(y=1000, color='r', linestyle='--', label='Target (1000/class)', alpha=0.7)
plt.xlabel('Class')
plt.ylabel('Number of Images')
plt.title('Distribution of Images by Class')
plt.xticks(range(len(df_count)), df_count['Class'], rotation=45, ha='right')
plt.legend()
plt.tight_layout()
plt.show()

## 2. Sample Images Visualization

In [ ]:
import random
from PIL import Image

def plot_random_samples(dataset_root, classes, samples_per_class=3):
    """Display random samples from each class using PIL"""
    # Only show classes that have images
    classes_with_images = []
    for cls in classes:
        path = dataset_root / cls
        if path.exists():
            images = [f for f in os.listdir(path) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))]
            if images:
                classes_with_images.append(cls)
    
    if not classes_with_images:
        print('No images found in dataset')
        return
    
    n_classes = len(classes_with_images)
    fig, axes = plt.subplots(n_classes, samples_per_class, figsize=(12, 2 * n_classes))
    if n_classes == 1:
        axes = axes.reshape(1, -1)
    
    for i, cls in enumerate(classes_with_images):
        path = dataset_root / cls
        all_images = [f for f in os.listdir(path) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))]
        
        if len(all_images) < samples_per_class:
            random_images = all_images
        else:
            random_images = random.sample(all_images, samples_per_class)
        
        for j, img_name in enumerate(random_images):
            img_path = path / img_name
            try:
                img = Image.open(img_path)
                axes[i, j].imshow(img)
            except Exception as e:
                axes[i, j].text(0.5, 0.5, f'Error: {str(e)[:15]}', 
                                ha='center', va='center', transform=axes[i, j].transAxes, fontsize=8)
            axes[i, j].axis('off')
            
            if j == 0:
                axes[i, j].set_title(cls, fontsize=12, loc='left', fontweight='bold')
    
    plt.tight_layout()
    plt.show()

print('Displaying random samples from each class...')
plot_random_samples(dataset_root, classes, samples_per_class=3)

## 3. Image Size Distribution

In [ ]:
# Scan all image sizes
widths, heights, labels = [], [], []

for cls in classes:
    path = dataset_root / cls
    if not path.exists():
        continue
    for img_name in os.listdir(path):
        if img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
            try:
                with Image.open(path / img_name) as img:
                    w, h = img.size
                    widths.append(w)
                    heights.append(h)
                    labels.append(cls)
            except Exception as e:
                print(f'Error reading {img_name}: {e}')
                continue

print(f'Scanned {len(widths)} images')

In [ ]:
if len(widths) > 0:
    df_sizes = pd.DataFrame({'Width': widths, 'Height': heights, 'Class': labels})
    
    # Scatter plot
    plt.figure(figsize=(12, 7))
    unique_classes = df_sizes['Class'].unique()
    import matplotlib.cm as cm
    import numpy as np
    colors_scatter = cm.rainbow(np.linspace(0, 1, len(unique_classes)))
    for i, cls in enumerate(unique_classes):
        cls_data = df_sizes[df_sizes['Class'] == cls]
        plt.scatter(cls_data['Width'], cls_data['Height'], 
                   label=cls, alpha=0.5, s=20, color=colors_scatter[i])
    
    plt.xlabel('Width (Pixels)')
    plt.ylabel('Height (Pixels)')
    plt.title('Image Size Distribution in Dataset')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Statistics
    print('\nImage size statistics (per class):')
    print('='*80)
    stats = df_sizes.groupby('Class')[['Width', 'Height']].agg(['min', 'max', 'mean', 'std']).round(0)
    print(stats)
else:
    print('No images found in dataset folders')

## 4. Preprocessing (Optional)

- Resize to standard size (e.g., 224x224)
- Normalize pixel values
- Save to `data/processed/`

In [ ]:
# Note: Preprocessing will be done here when needed
# For now, training can use raw images with on-the-fly augmentation

print('Preprocessing to be implemented...')
print('Standard size: 224x224 or 256x256')
print('Normalization: ImageNet mean/std or 0-1 scaling')